# Coordinate Systems Demo

This notebook demonstrates skyvista's support for different coordinate systems:

1. **Geographic (lat/lon)** - Data on Earth's surface, converted to 3D Cartesian on a sphere
2. **Spherical (radar)** - Range/azimuth/elevation data, converted to Cartesian centered at instrument

In [1]:
import numpy as np
import xarray as xr
import skyvista as sv
import pyvista as pv

## 1. Geographic Coordinates (Lat/Lon)

Geographic data uses latitude, longitude, and altitude. Skyvista automatically detects these coordinates and converts them to 3D Cartesian coordinates on a sphere.

The conversion formula:
- `x = (R + alt) * cos(lat) * cos(lon)`
- `y = (R + alt) * cos(lat) * sin(lon)`
- `z = (R + alt) * sin(lat)`

In [2]:
# Create toy geographic data
# A regional domain over the central US
lon = np.linspace(-105, -95, 50)  # degrees
lat = np.linspace(30, 40, 50)     # degrees
alt = np.linspace(0, 15000, 30)   # meters (0 to 15 km)

# Create a 3D temperature field with some structure
LON, LAT, ALT = np.meshgrid(lon, lat, alt, indexing='ij')

# Temperature decreases with altitude, with a warm anomaly
temperature = 300 - ALT / 500  # Basic lapse rate
# Add a warm bubble
lon_center, lat_center, alt_center = -100, 35, 5000
dist = np.sqrt((LON - lon_center)**2 + (LAT - lat_center)**2 + ((ALT - alt_center)/1000)**2)
temperature += 10 * np.exp(-dist**2 / 10)

# Create xarray Dataset
geo_ds = xr.Dataset(
    {
        'temperature': (['lon', 'lat', 'altitude'], temperature),
    },
    coords={
        'lon': lon,
        'lat': lat,
        'altitude': alt,
    }
)

print(geo_ds)
print(f"\nIs geographic grid: {sv.is_geographic_grid(geo_ds, sv.resolve_coordinates(geo_ds, ['x', 'y', 'z']))}")

<xarray.Dataset> Size: 601kB
Dimensions:      (lon: 50, lat: 50, altitude: 30)
Coordinates:
  * lon          (lon) float64 400B -105.0 -104.8 -104.6 ... -95.41 -95.2 -95.0
  * lat          (lat) float64 400B 30.0 30.2 30.41 30.61 ... 39.59 39.8 40.0
  * altitude     (altitude) float64 240B 0.0 517.2 ... 1.448e+04 1.5e+04
Data variables:
    temperature  (lon, lat, altitude) float64 600kB 300.0 299.0 ... 271.0 270.0

Is geographic grid: True


/home/cmdavis4/projects/carlee/skyvista/skyvista/grids.py:189: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  available = set(ds.coords.keys()) | set(ds.dims.keys())


In [3]:
# Check what grid type is detected
builder = sv.detect_grid_type(geo_ds)
print(f"Detected grid type: {builder.grid_type}")
print(f"Builder class: {type(builder).__name__}")

Detected grid type: geographic
Builder class: GeographicGridBuilder


In [4]:
# Build the mesh and visualize
# Note: The default Earth radius makes the altitude variations very small
# compared to Earth's radius, so we'll use altitude_scale to exaggerate

# Create a custom builder with altitude exaggeration
geo_builder = sv.GeographicGridBuilder(
    lon_coord='lon',
    lat_coord='lat', 
    alt_coord='altitude',
    earth_radius=sv.EARTH_RADIUS_M,
    altitude_scale=100.0,  # Exaggerate altitude 100x for visibility
)

mesh = geo_builder.build_mesh(geo_ds, varname='temperature')
print(f"Mesh type: {type(mesh).__name__}")
print(f"Mesh bounds: {mesh.bounds}")
print(f"Number of points: {mesh.n_points}")

Mesh type: StructuredGrid
Mesh bounds: BoundsTuple(x_min = -1764236.385358688,
            x_max =  -425360.9134734901,
            y_min = -6790547.1661820635,
            y_max = -4714171.193601506,
            z_min =  3185499.9999999995,
            z_max =  5059381.27584275)
Number of points: 75000


In [6]:
# Visualize the geographic data
plotter = pv.Plotter()

# Add isosurface of temperature
contour = mesh.contour(isosurfaces=[295, 300, 305], scalars='temperature')
plotter.add_mesh(contour, scalars='temperature', cmap='coolwarm', opacity=0.7)

# Add a reference sphere at Earth's surface
# sphere = pv.Sphere(radius=sv.EARTH_RADIUS_M, theta_resolution=50, phi_resolution=50)
# plotter.add_mesh(sphere, color='lightblue', opacity=0.3)

plotter.add_text("Geographic Coordinates (lat/lon)\nAltitude exaggerated 100x", font_size=10)
plotter.show()

Widget(value='<iframe src="http://localhost:46635/index.html?ui=P_0x7fc4bd289bd0_1&reconnect=auto" class="pyvi…

## 2. Spherical/Radar Coordinates

Radar data uses range (distance from radar), azimuth (horizontal angle), and elevation (vertical angle). Skyvista converts these to Cartesian coordinates centered at the radar location.

The conversion formula (radar convention):
- `x = range * cos(elevation) * sin(azimuth)`
- `y = range * cos(elevation) * cos(azimuth)`
- `z = range * sin(elevation)`

Where azimuth is measured clockwise from north (y-axis).

In [7]:
# Create toy radar data
# Typical radar scan parameters
range_km = np.linspace(0.5, 150, 100)  # 0.5 to 150 km range
azimuth = np.linspace(0, 359, 360)     # Full 360 degree sweep
elevation = np.array([0.5, 1.5, 2.5, 3.5, 4.5])  # 5 elevation tilts

# Create a 3D reflectivity field
RANGE, AZ, EL = np.meshgrid(range_km, azimuth, elevation, indexing='ij')

# Create a storm cell pattern
# Storm center at ~50km range, 225 degrees azimuth
storm_range = 50
storm_az = 225

# Convert to Cartesian for distance calculation
x = RANGE * np.cos(np.deg2rad(EL)) * np.sin(np.deg2rad(AZ))
y = RANGE * np.cos(np.deg2rad(EL)) * np.cos(np.deg2rad(AZ))
z = RANGE * np.sin(np.deg2rad(EL))

storm_x = storm_range * np.sin(np.deg2rad(storm_az))
storm_y = storm_range * np.cos(np.deg2rad(storm_az))

dist_from_storm = np.sqrt((x - storm_x)**2 + (y - storm_y)**2 + z**2)

# Reflectivity pattern: high near storm center, decreasing outward
reflectivity = 60 * np.exp(-dist_from_storm**2 / 400) - 10
reflectivity = np.clip(reflectivity, -10, 60)  # Typical dBZ range

# Create xarray Dataset
radar_ds = xr.Dataset(
    {
        'reflectivity': (['range', 'azimuth', 'elevation'], reflectivity),
    },
    coords={
        'range': range_km * 1000,  # Convert to meters
        'azimuth': azimuth,
        'elevation': elevation,
    }
)

print(radar_ds)
print(f"\nIs spherical grid: {sv.is_spherical_grid(radar_ds)}")

<xarray.Dataset> Size: 1MB
Dimensions:       (range: 100, azimuth: 360, elevation: 5)
Coordinates:
  * range         (range) float64 800B 500.0 2.01e+03 ... 1.485e+05 1.5e+05
  * azimuth       (azimuth) float64 3kB 0.0 1.0 2.0 3.0 ... 357.0 358.0 359.0
  * elevation     (elevation) float64 40B 0.5 1.5 2.5 3.5 4.5
Data variables:
    reflectivity  (range, azimuth, elevation) float64 1MB -9.894 ... -10.0

Is spherical grid: True


In [8]:
# Check what grid type is detected
builder = sv.detect_grid_type(radar_ds)
print(f"Detected grid type: {builder.grid_type}")
print(f"Builder class: {type(builder).__name__}")

# Check resolved coordinates
coords = sv.resolve_spherical_coordinates(radar_ds)
print(f"Resolved spherical coordinates: {coords}")

Detected grid type: spherical
Builder class: SphericalGridBuilder
Resolved spherical coordinates: {'range': 'range', 'azimuth': 'azimuth', 'elevation': 'elevation'}


In [9]:
# Build the mesh
radar_mesh = builder.build_mesh(radar_ds, varname='reflectivity')
print(f"Mesh type: {type(radar_mesh).__name__}")
print(f"Mesh bounds: {radar_mesh.bounds}")
print(f"Number of points: {radar_mesh.n_points}")

Mesh type: StructuredGrid
Mesh bounds: BoundsTuple(x_min = -149994.2884596257,
            x_max =  149994.2884596257,
            y_min = -149994.2884596257,
            y_max =  149994.2884596257,
            z_min =       4.363267749186967,
            z_max =   11768.864359176741)
Number of points: 180000


In [11]:
# Visualize the radar data
plotter = pv.Plotter()

# Add isosurfaces of reflectivity at typical thresholds
contour = radar_mesh.contour(isosurfaces=[20, 35, 50], scalars='reflectivity')
plotter.add_mesh(
    contour, 
    scalars='reflectivity', 
    # cmap='pyart_NWSRef',  # Radar reflectivity colormap if available
    clim=[-10, 60],
    opacity=0.7
)

# Add radar location marker
radar_marker = pv.Sphere(radius=2000, center=(0, 0, 0))
plotter.add_mesh(radar_marker, color='red')

# Add ground plane for reference
ground = pv.Plane(center=(0, 0, 0), direction=(0, 0, 1), i_size=300000, j_size=300000)
plotter.add_mesh(ground, color='green', opacity=0.2)

plotter.add_text("Radar Coordinates (range/azimuth/elevation)\nStorm cell at ~50km, 225 deg", font_size=10)
plotter.show_grid()
plotter.show()

Widget(value='<iframe src="http://localhost:46635/index.html?ui=P_0x7fc4bd1aa710_3&reconnect=auto" class="pyvi…

## 3. Using the Scene API

The Scene API automatically detects coordinate types and handles the conversion internally.

In [ ]:
# Using Scene with radar data
scene = sv.Scene(background='#1a1a2e')
scene.add_contour(
    radar_ds, 
    'reflectivity', 
    isosurfaces=[25, 40, 55],
    opacity=0.6,
    cmap='hot'
)
scene.show()

## 4. Radar Data with Multiple Dimensions

A more realistic radar scenario where azimuth and elevation vary with volume scan and time.

In [ ]:
# Create radar data with (range, volume_scan, time) structure
# where azimuth and elevation are (volume_scan, time)
n_range = 50
n_volume_scan = 10  # 10 elevation sweeps per volume
n_time = 5  # 5 time steps

range_vals = np.linspace(1000, 100000, n_range)  # 1-100 km

# Azimuth increases with volume scan (simulating PPI scans)
# and cycles through time
az_base = np.linspace(0, 350, n_volume_scan)
time_offset = np.arange(n_time) * 36  # Each time step rotates 36 degrees
azimuth_2d = (az_base[:, np.newaxis] + time_offset[np.newaxis, :]) % 360

# Elevation increases with volume scan (typical VCP pattern)
el_base = np.array([0.5, 1.5, 2.4, 3.4, 4.3, 5.3, 6.2, 7.5, 8.7, 10.0])
elevation_2d = np.broadcast_to(el_base[:, np.newaxis], (n_volume_scan, n_time))

# Create reflectivity data
reflectivity = np.random.randn(n_range, n_volume_scan, n_time) * 10 + 20
reflectivity = np.clip(reflectivity, -10, 60)

radar_ds_2d = xr.Dataset(
    {
        'reflectivity': (['range', 'volume_scan', 'time'], reflectivity),
    },
    coords={
        'range': range_vals,
        'azimuth': (['volume_scan', 'time'], azimuth_2d),
        'elevation': (['volume_scan', 'time'], elevation_2d.copy()),
        'volume_scan': np.arange(n_volume_scan),
        'time': np.arange(n_time),
    }
)

print(radar_ds_2d)
print(f"\nIs spherical grid: {sv.is_spherical_grid(radar_ds_2d)}")
print(f"Azimuth shape: {radar_ds_2d['azimuth'].shape}")
print(f"Elevation shape: {radar_ds_2d['elevation'].shape}")

In [ ]:
# Build and visualize
builder = sv.detect_grid_type(radar_ds_2d)
print(f"Detected: {builder.grid_type}")

mesh = builder.build_mesh(radar_ds_2d, varname='reflectivity')
print(f"Mesh shape: {mesh.dimensions}")

plotter = pv.Plotter()
plotter.add_mesh(mesh, scalars='reflectivity', cmap='viridis', opacity=0.5)
plotter.add_mesh(pv.Sphere(radius=1000, center=(0,0,0)), color='red')
plotter.show_grid()
plotter.add_text("Radar with 2D azimuth/elevation\n(volume_scan, time)", font_size=10)
plotter.show()

## 5. Explicitly Specifying Grid Type

You can explicitly specify the grid type if auto-detection doesn't work for your data.

In [ ]:
# Force geographic interpretation even if coordinates aren't named conventionally
weird_ds = geo_ds.rename({'lon': 'x_coord', 'lat': 'y_coord', 'altitude': 'z_coord'})
# Add attributes to help detection
weird_ds['x_coord'].attrs['units'] = 'degrees_east'
weird_ds['y_coord'].attrs['units'] = 'degrees_north'

# This should still detect as geographic due to units
builder = sv.detect_grid_type(weird_ds)
print(f"Detected: {builder.grid_type}")

# Or force it explicitly
builder = sv.get_grid_builder(weird_ds, grid_type='geographic')
print(f"Forced: {builder.grid_type}")

## Summary

Skyvista supports multiple coordinate systems:

| Grid Type | Coordinates | Use Case |
|-----------|-------------|----------|
| `rectilinear` | x, y, z (1D, Cartesian) | Model output in meters |
| `curvilinear` | x, y, z (2D/3D) | Non-orthogonal model grids |
| `geographic` | lon, lat, altitude | Earth-based data |
| `spherical` | range, azimuth, elevation | Radar data |
| `unstructured` | x, y, z (point cloud) | Particle/observation data |

Detection is automatic based on coordinate names and CF attributes, but can be overridden with `grid_type` parameter.